In [24]:
import matplotlib.pyplot as plt 
import torch
import flappy_bird_gymnasium
import gymnasium as gym
from ppo import PPO_Clip
from my_networks import ActorNN

In [25]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [26]:
env = gym.make("FlappyBird-v0", use_lidar=False)
print(f"Action Dim: {env.action_space.n}")
print(f"Obs Dim: {env.observation_space}")

Action Dim: 2
Obs Dim: Box(-1.0, 1.0, (12,), float64)


In [ ]:
# Crear agente PPO
agent = PPO_Clip(env)

# Entrenar durante cierto número de timesteps
rewards = agent.learn(total_timesteps=1_000_000)

# Guardar modelo
torch.save(agent.actor.state_dict(), "flappy_actor.pth")
torch.save(agent.critic.state_dict(), "flappy_critic.pth")

Timestep: 0 | Reward promedio: -7.28
Timestep: 2050 | Reward promedio: -6.49
Timestep: 4100 | Reward promedio: -5.89
Timestep: 6150 | Reward promedio: -4.97
Timestep: 8200 | Reward promedio: -3.26
Timestep: 10250 | Reward promedio: -1.09
Timestep: 12312 | Reward promedio: 1.07
Timestep: 14362 | Reward promedio: 2.90
Timestep: 16443 | Reward promedio: 3.45
Timestep: 18527 | Reward promedio: 3.93
Timestep: 20587 | Reward promedio: 4.01
Timestep: 22642 | Reward promedio: 3.64
Timestep: 24706 | Reward promedio: 3.66
Timestep: 26782 | Reward promedio: 4.45
Timestep: 28873 | Reward promedio: 3.86
Timestep: 30944 | Reward promedio: 4.35
Timestep: 32994 | Reward promedio: 4.33
Timestep: 35088 | Reward promedio: 4.43
Timestep: 37183 | Reward promedio: 4.63
Timestep: 39250 | Reward promedio: 4.99
Timestep: 41414 | Reward promedio: 4.95
Timestep: 43512 | Reward promedio: 4.89
Timestep: 45580 | Reward promedio: 5.22
Timestep: 47675 | Reward promedio: 5.64
Timestep: 49773 | Reward promedio: 5.22
Ti

In [29]:
# ====== CONFIGURAR ENTORNO ======
env = gym.make("FlappyBird-v0", render_mode="human", use_lidar=False)

obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

# ====== CARGAR POLÍTICA ======
actor = ActorNN(obs_dim, n_actions)
actor.load_state_dict(torch.load("flappy_actor.pth", map_location=torch.device("cpu")))
actor.eval()

# ====== EVALUAR POLÍTICA ======
n_episodes = 5
for ep in range(n_episodes):
    obs, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        # --- convertir obs a tensor 1D ---
        obs_t = torch.tensor(obs, dtype=torch.float32)

        # --- inferencia sin gradientes ---
        with torch.no_grad():
            logits = actor(obs_t)

            # si hay batch extra, eliminarla
            if logits.dim() > 1:
                logits = logits.squeeze(0)

            # acción determinista (mayor activación ReLU)
            action = torch.argmax(logits).item()

        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward

    print(f"Episode {ep + 1}: total reward = {total_reward}")

env.close()

Episode 1: total reward = 2.0000000000000013
Episode 2: total reward = 2.0000000000000013
Episode 3: total reward = 2.0000000000000013
Episode 4: total reward = 2.0000000000000013
Episode 5: total reward = 2.0000000000000013
